# Assignment 3 — End-to-end multimodal fine-tuning

This is a joint fine-tune of ResNet-50 and BERT-base with the GMU classifier.

- Encoder LRs: 1e-5 each.
- GMU LR: 1e-4.
- Mixed precision.
- Checkpoint: best dev macro-F1.
- Missing-modality checks: image dropped vs text dropped.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "assignment3").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "assignment3" / "src"))

import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer

from utils.mmimdb import (
    build_label_matrix,
    build_splits,
    get_genres,
    load_mmimdb,
    make_multimodal_loader,
)
from models import TEXT_MODEL_NAME, build_multimodal_gmu
from training_loop import fit_multimodal, metrics_from_probabilities, run_multimodal_epoch

SEED = 42
EPOCHS = 3
BATCH_SIZE = 64
WORKERS = 8
IMAGE_LR = 1e-5
TEXT_LR = 1e-5
FUSION_LR = 1e-4
LIMIT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

RESULTS_DIR = ROOT / "assignment3" / "results"
CHECKPOINT_DIR = ROOT / "assignment3" / "checkpoints"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.BCEWithLogitsLoss()
print("device:", DEVICE)

device: cuda


In [2]:
dataset = load_mmimdb()
genres = get_genres(dataset)
labels = build_label_matrix(dataset, genres)
splits = build_splits(dataset, genres)

if LIMIT:
    splits = {name: indices[:LIMIT] for name, indices in splits.items()}

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
loaders = {
    name: make_multimodal_loader(
        dataset,
        indices,
        labels,
        tokenizer,
        BATCH_SIZE,
        name == "train",
        WORKERS,
    )
    for name, indices in splits.items()
}
print("splits:", {name: len(indices) for name, indices in splits.items()})

splits: {'train': 15558, 'dev': 2595, 'test': 7806}


In [3]:
model = build_multimodal_gmu(
    num_labels=len(genres),
    hidden_dim=512,
    modality_dropout=0.2,
).to(DEVICE)
history, best_epoch, thresholds = fit_multimodal(
    model,
    loaders,
    DEVICE,
    epochs=EPOCHS,
    image_lr=IMAGE_LR,
    text_lr=TEXT_LR,
    fusion_lr=FUSION_LR,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "finetune_gmu.pt",
)
print("best epoch:", best_epoch)

epoch 01/3 | train loss 0.3003 | val loss 0.2091 macro-F1 0.4909 | 39.8s


epoch 02/3 | train loss 0.2175 | val loss 0.1791 macro-F1 0.5868 | 37.7s


epoch 03/3 | train loss 0.1940 | val loss 0.1694 macro-F1 0.6118 | 37.6s
best epoch: 3


In [4]:
_, test_probs, test_targets = run_multimodal_epoch(
    model, loaders["test"], criterion, DEVICE, AMP
)
full_metrics = metrics_from_probabilities(test_probs, test_targets, thresholds)
print("full:", {key: full_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

missing_metrics = {}
for mode in ("image", "text"):
    _, probs, targets = run_multimodal_epoch(
        model, loaders["test"], criterion, DEVICE, AMP, drop_mode=mode
    )
    missing_metrics[mode] = metrics_from_probabilities(probs, targets, thresholds)
    print(mode, {key: missing_metrics[mode][key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

full: {'macro_f1': 0.5956771409463218, 'micro_f1': 0.6699478149036678, 'samples_f1': 0.6690127562533404, 'exact_match': 0.19446579554189086}


image {'macro_f1': 0.5824122871760631, 'micro_f1': 0.653992960888316, 'samples_f1': 0.6552098362820884, 'exact_match': 0.18472969510632847}


text {'macro_f1': 0.20415068801884326, 'micro_f1': 0.4554799164655149, 'samples_f1': 0.44296505975022504, 'exact_match': 0.07750448373046374}


In [5]:
row = {
    "model": "End-to-end multimodal GMU fine-tune",
    "params": sum(p.numel() for p in model.parameters()),
    "best_epoch": int(best_epoch),
    "train_time_s": float(sum(history["epoch_time"])),
    "test_macro_f1": full_metrics["macro_f1"],
    "test_micro_f1": full_metrics["micro_f1"],
    "test_samples_f1": full_metrics["samples_f1"],
    "test_exact_match": full_metrics["exact_match"],
}
results_df = pd.DataFrame([row])
results_df

payload = {
    "config": {
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_lr": IMAGE_LR,
        "text_lr": TEXT_LR,
        "fusion_lr": FUSION_LR,
        "amp": AMP,
        "limit": LIMIT,
    },
    "results": [row],
    "missing_metrics": missing_metrics,
    "thresholds": thresholds.tolist(),
    "history": history,
}
with open(RESULTS_DIR / "m3_finetune_gmu.json", "w") as handle:
    json.dump(payload, handle, indent=2)
results_df.to_csv(RESULTS_DIR / "m3_finetune_gmu.csv", index=False)
print("saved", RESULTS_DIR / "m3_finetune_gmu.json")

saved /raid/home/an.nv/main/deep-learning/CO3133-DeepLearning/assignment3/results/m3_finetune_gmu.json
